In [9]:
!pip install -q transformers datasets accelerate torch

In [10]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

In [11]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)


model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
import pandas as pd

true_df = pd.read_csv("True.csv")
fake_df = pd.read_csv("Fake.csv")

true_df["label"] = 1   # REAL
fake_df["label"] = 0   # FAKE

df = pd.concat([true_df, fake_df], axis=0)
df = df.sample(frac=1).reset_index(drop=True)

df = df[["text", "label"]]
df.head()

,text,label
0,Donald Trump has repeatedly accused Democrats ...,0
1,"SIMFEROPOL, Crimea (Reuters) - A man who led p...",1
2,Israel is in a class all by itself when it com...,0
3,"Because according to the left, the brave men w...",0
4,LONDON (Reuters) - Britain s Prince Harry and ...,1


In [5]:
def clean_text(text):
    text = text.lower()
    return text

df["text"] = df["text"].apply(clean_text)

In [6]:
df["text"] = df["text"].str.lower()

In [7]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)
dataset


Dataset({
    features: ['text', 'label'],
    num_rows: 44898
})

In [12]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset


Map:   0%|          | 0/44898 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 44898
})

In [13]:
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2)

train_dataset = tokenized_dataset["train"]
test_dataset = tokenized_dataset["test"]

In [14]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True,
    report_to="none"
)


In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer
)


/tmp/ipython-input-1929025085.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [16]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.000000,0.001187
2,0.002900,0.000577


TrainOutput(global_step=8980, training_loss=0.004336043876476009, metrics={'train_runtime': 3828.8176, 'train_samples_per_second': 18.762, 'train_steps_per_second': 2.345, 'total_flos': 9515928049852416.0, 'train_loss': 0.004336043876476009, 'epoch': 2.0})

In [17]:
trainer.evaluate()

{'eval_loss': 0.0005773992161266506,
 'eval_runtime': 135.4209,
 'eval_samples_per_second': 66.312,
 'eval_steps_per_second': 8.293,
 'epoch': 2.0}

In [19]:
model.save_pretrained("fake_news_model")
tokenizer.save_pretrained("fake_news_tokenizer")

('fake_news_tokenizer/tokenizer_config.json',
 'fake_news_tokenizer/special_tokens_map.json',
 'fake_news_tokenizer/vocab.txt',
 'fake_news_tokenizer/added_tokens.json',
 'fake_news_tokenizer/tokenizer.json')

In [22]:
# Test the prediction
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="fake_news_model",
    tokenizer="fake_news_tokenizer"
)

classifier("Breaking: Government confirms alien invasion")

Device set to use cuda:0


[{'label': 'LABEL_0', 'score': 0.9999915361404419}]